### Import Packages

In [ ]:
import os

import glob

import pandas as pd

import zipfile
from pathlib import Path


import numpy as np
import pickle

### Notebook Settings

In [ ]:
# Display Settings for Pandas DataFrames
pd.set_option("display.max_rows", None)         # How many rows to display when printing a DataFrame? None shows all rows.
pd.set_option("display.max_columns", None)      # How many columns to display when printing a DataFrame? None shows all columns.
pd.set_option("display.max_info_columns", 100)  # How many rows to display for .info()?
pd.set_option("display.max_info_rows", 1000000) # How many columns to display for .info()?
pd.set_option("display.precision", 2)           # How many decimals to show when printing a DataFrame?

### Load "Prepared" Parquet File

In [ ]:
# Load the prepared parquet file
df_prov_imp = pd.read_parquet("1.0-prepared.parquet")

# Display basic info
print(f"Shape: {df_prov_imp.shape}")
print(f"\nColumns: {df_prov_imp.columns.tolist()}")
print(f"\nFirst few rows:")
df_prov_imp.head()

### Feature Selection

This cell selects the relevant columns from the full dataset and stores them in `df_prov_imp`. The selected features cover four categories:

- **Demographics** — `Age Band`, `Gender`
- **Pre-op survey responses** — patient-reported symptom duration, prior surgery, living arrangements, and disability status
- **Comorbidities** — 10 binary flags indicating chronic conditions (e.g. heart disease, hypertension, diabetes, cancer)
- **Oxford Knee Score (OKS) — Pre-op items** — 11 functional/pain questions (e.g. walking, kneeling, stairs) plus the composite pre-op score

The **outcome variable** is health gain, defined as `Knee Replacement Post-Op Q Score` - `Knee Replacement Pre-Op Q Score`, which measures patient-reported health gain after surgery. All other columns in the original dataset are dropped.

In [ ]:
keep_cols = [
    "Age Band",
    "Gender",
    "Pre-Op Q Symptom Period",
    "Pre-Op Q Previous Surgery",
    "Pre-Op Q Living Arrangements",
    "Pre-Op Q Disability",
    "Heart Disease",
    "High Bp",
    "Stroke",
    "Circulation",
    "Lung Disease",
    "Diabetes",
    "Kidney Disease",
    "Nervous System",
    "Liver Disease",
    "Cancer",
    "Depression",
    "Arthritis",
    "Knee Replacement Pre-Op Q Pain",
    "Knee Replacement Pre-Op Q Night Pain",
    "Knee Replacement Pre-Op Q Washing",
    "Knee Replacement Pre-Op Q Transport",
    "Knee Replacement Pre-Op Q Walking",
    "Knee Replacement Pre-Op Q Standing",
    "Knee Replacement Pre-Op Q Limping",
    "Knee Replacement Pre-Op Q Kneeling",
    "Knee Replacement Pre-Op Q Work",
    "Knee Replacement Pre-Op Q Confidence",
    "Knee Replacement Pre-Op Q Shopping",
    "Knee Replacement Pre-Op Q Stairs",
    "Knee Replacement Pre-Op Q Score",
    "Knee Replacement Post-Op Q Score",
]

df_prov_slc = df_prov_imp[keep_cols].copy()

In [ ]:
df_prov_slc.shape

(139236, 32)

### Missingness

This cell audits missing data across the selected feature columns in `df_prov_imp`.

For each column it computes:
- **`missing`** — count of `NaN` values *or* NHS suppression markers (`*`), which were not caught by the earlier standardisation step
- **`complete_pct`** — percentage of rows that are non-missing

The resulting summary table `df_prov_imp_missing` is sorted ascending by completeness so the most problematic columns appear first.

This informs downstream decisions about imputation strategy: columns with low completeness may need a different approach than near-complete ones.

In [ ]:
# Pandas.
df_prov_slc_missing = (
    
    pd.DataFrame({

        'data_type':    df_prov_slc.dtypes,
        'missing':      (df_prov_slc.isnull() | df_prov_slc.eq('*')).sum()   
    })

    # Add complete%
    .assign(
        complete_pct = lambda x: round(100 * (n_obs - x['missing']) / n_obs, 2)
    )

    # Remove variables that are complete.
    #.query("complete_pct < 100")

    # Sort table by number of missing data in ascending order.
    .sort_values(
        by        = 'complete_pct',
        ascending = True
    )
)

df_prov_slc_missing

,data_type,missing,complete_pct
Age Band,object,9402,93.25
Gender,object,9402,93.25
Pre-Op Q Disability,float64,5907,95.76
Knee Replacement Pre-Op Q Score,float64,5517,96.04
Knee Replacement Post-Op Q Score,float64,2960,97.87
Pre-Op Q Living Arrangements,float64,2087,98.50
Knee Replacement Pre-Op Q Standing,float64,1453,98.96
Knee Replacement Pre-Op Q Walking,float64,1443,98.96
Knee Replacement Pre-Op Q Limping,float64,1415,98.98
Knee Replacement Pre-Op Q Kneeling,float64,1396,99.00


### Scoping (Age, Gender, etc)

Remove Unknown Age Bands

In [ ]:
dem_cols = [
    "Age Band",
    "Gender"
]

# Replace "*" with NaN to treat as missing values
df_prov_slc[dem_cols] = df_prov_slc[dem_cols].replace("*", None)

# Drop rows with missing age and gender
df_prov_slc = df_prov_slc.dropna(subset=dem_cols)
print(f"Shape of the dataframe after dropping rows with missing age and gender: {df_prov_slc.shape}")

Shape of the dataframe after dropping rows with missing age and gender: (129834, 32)


In [ ]:
# Count patients by Age, sorted by value
print(df_prov_slc["Age Band"].value_counts().sort_index())

# Count all patients
df_prov_slc["Age Band"].value_counts().sum()

Age Band
40 to 49       250
50 to 59     13260
60 to 69     45361
70 to 79     55056
80 to 89     15883
90 to 120       24
Name: count, dtype: int64


np.int64(129834)

In [ ]:
# Count patients by Age, sorted by value
print(df_prov_slc["Gender"].value_counts().sort_index())

# Count all patients
df_prov_slc["Gender"].value_counts().sum()

Gender
1    55749
2    74085
Name: count, dtype: int64


np.int64(129834)

Delete Age Bands 90 to 120?

Verify that all Comorbidity is complete

### Remove all Missing Pre-Op and Post-Op Missing Scores

In [ ]:
# Count patients by OKS Pre-Op Dimensions, sorted by value
for col in ["Knee Replacement Pre-Op Q Pain",
    "Knee Replacement Pre-Op Q Night Pain",
    "Knee Replacement Pre-Op Q Washing",
    "Knee Replacement Pre-Op Q Transport",
    "Knee Replacement Pre-Op Q Walking",
    "Knee Replacement Pre-Op Q Standing",
    "Knee Replacement Pre-Op Q Limping",
    "Knee Replacement Pre-Op Q Kneeling",
    "Knee Replacement Pre-Op Q Work",
    "Knee Replacement Pre-Op Q Confidence",
    "Knee Replacement Pre-Op Q Shopping",
    "Knee Replacement Pre-Op Q Stairs",
    "Knee Replacement Pre-Op Q Score",
    "Knee Replacement Post-Op Q Score"]:
    print(f"\n{col}:")
    print(df_prov_slc[col].value_counts().sort_index())


Knee Replacement Pre-Op Q Pain:
Knee Replacement Pre-Op Q Pain
0.0    65650
1.0    56745
2.0     5731
3.0     1229
4.0      298
Name: count, dtype: int64

Knee Replacement Pre-Op Q Night Pain:
Knee Replacement Pre-Op Q Night Pain
0.0    41840
1.0    38259
2.0    33570
3.0     6875
4.0     8064
Name: count, dtype: int64

Knee Replacement Pre-Op Q Washing:
Knee Replacement Pre-Op Q Washing
0.0      735
1.0    11274
2.0    40735
3.0    35590
4.0    41386
Name: count, dtype: int64

Knee Replacement Pre-Op Q Transport:
Knee Replacement Pre-Op Q Transport
0.0      684
1.0    29739
2.0    67384
3.0    22286
4.0     8466
Name: count, dtype: int64

Knee Replacement Pre-Op Q Walking:
Knee Replacement Pre-Op Q Walking
0.0    15755
1.0    16256
2.0    52724
3.0    33448
4.0    10291
Name: count, dtype: int64

Knee Replacement Pre-Op Q Standing:
Knee Replacement Pre-Op Q Standing
0.0     4399
1.0    55950
2.0    47471
3.0    18291
4.0     2354
Name: count, dtype: int64

Knee Replacement Pre-Op Q L

### Compare Imputed vs Non-Imputed DataFrames

In [ ]:
# Compare Results
pd.DataFrame({
    "original": df_prov_slc['Arthritis'].describe(),
    "imputed": df_prov_slc['Arthritis'].describe()
})

,original,imputed
count,129834.00,129834.00
mean,0.77,0.77
std,0.42,0.42
min,0.00,0.00
25%,1.00,1.00
50%,1.00,1.00
75%,1.00,1.00
max,1.00,1.00


### Calculate Health Gain